# Total Energy K-Grid Convergence

Run a total-energy k-grid convergence workflow for a material on the Mat3ra platform and return the converged k-grid for reuse in another notebook.

<h2 style="color:green">Usage</h2>

1. Set material, workflow, convergence, and compute parameters in cell 1.2. below (or use the default values).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to review the convergence series and the converged k-grid.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure parameters for the material, workflow, convergence loop, compute resources, and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Create material: materials are read from the `../uploads` folder. If the material is not found by name, Standata is used as a fallback. The material is then saved to the platform.
1. Create workflow and add convergence: select the application, load the total energy workflow from Standata, inject the convergence loop into the workflow in the notebook, and inspect the resulting workflow graph.
1. Configure compute: get the list of clusters and create compute configuration with the selected cluster, queue, and number of processors.
1. Create the job with material and workflow configuration: assemble the job from the saved material, convergence-enabled workflow, project, and compute configuration.
1. Submit the job and monitor the status: submit the job and wait for completion.
1. Retrieve results: reconstruct the convergence series from `job.scopeTrack`, and extract the converged k-grid for reuse elsewhere.


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)


In [ ]:
import sys

if sys.platform == "emscripten":
    import micropip

    await micropip.install("mat3ra-api-examples", deps=False)
    await micropip.install("mat3ra-utils")
    from mat3ra.utils.jupyterlite.packages import install_packages

    await install_packages("api_examples")


### 1.2. Set parameters and configurations for the convergence job


In [ ]:
from datetime import datetime
from mat3ra.esse.models.workflow.subworkflow.convergence.enum_options import ConvergenceParameterNameEnum
from mat3ra.ide.compute import QueueName

# 1. Organization / account selection
ORGANIZATION_NAME = None

# 2. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"

# 3. Workflow parameters
APPLICATION_NAME = "espresso"
WORKFLOW_SEARCH_TERM = "total_energy.json"
MY_WORKFLOW_NAME = "Total Energy Convergence"

# 4. Convergence parameters
KGRID_CONVERGENCE_TYPE = ConvergenceParameterNameEnum.N_k  # N_k, N_k_nonuniform, or N_k_nonuniform_2D
UNIFORM_KGRID_INITIAL = 1
UNIFORM_KGRID_INCREMENT = 1
NON_UNIFORM_KGRID_INITIAL = [2, 2, 1]
NON_UNIFORM_KGRID_INCREMENT = 1
NON_UNIFORM_2D_KGRID_INITIAL = [2, 2]
NON_UNIFORM_2D_KGRID_INCREMENT = 1
RESULT_INITIAL = 0
TOLERANCE = 1e-3

# 5. Compute parameters
CLUSTER_NAME = None
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job naming
RUN_LABEL = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and store the credentials in the current environment.


In [ ]:
from utils.auth import authenticate

await authenticate()

### 2.2. Initialize API Client


In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project


In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
PROJECT_ID = projects[0]["_id"]
print(f"Using project: {projects[0]['name']} ({PROJECT_ID})")


## 3. Load and save the material
### 3.1. Load material from local uploads or Standata


In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from utils.jupyterlite import load_material_from_folder
from utils.visualize import visualize_materials as visualize

material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME)
)

visualize(material)


### 3.2. Save material to the platform


In [ ]:
from utils.api import get_or_create_material

saved_material_response = get_or_create_material(client, material, ACCOUNT_ID)
saved_material = Material.create(saved_material_response)
print(f"Saved material ID: {saved_material.id}")


## 4. Build the convergence workflow
### 4.1. Load the total-energy workflow


In [ ]:
from mat3ra.ade.application import Application
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from utils.visualize import visualize_workflow

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = f"{MY_WORKFLOW_NAME} {RUN_LABEL}"

visualize_workflow(workflow)


### 4.2. Add convergence directly in the notebook


In [ ]:
if KGRID_CONVERGENCE_TYPE == ConvergenceParameterNameEnum.N_k:
    PARAMETER_INITIAL = UNIFORM_KGRID_INITIAL
    PARAMETER_INCREMENT = UNIFORM_KGRID_INCREMENT
elif KGRID_CONVERGENCE_TYPE == ConvergenceParameterNameEnum.N_k_nonuniform:
    PARAMETER_INITIAL = NON_UNIFORM_KGRID_INITIAL
    PARAMETER_INCREMENT = NON_UNIFORM_KGRID_INCREMENT
elif KGRID_CONVERGENCE_TYPE == ConvergenceParameterNameEnum.N_k_nonuniform_2D:
    PARAMETER_INITIAL = NON_UNIFORM_2D_KGRID_INITIAL
    PARAMETER_INCREMENT = NON_UNIFORM_2D_KGRID_INCREMENT

CONVERGENCE_PARAMETER = KGRID_CONVERGENCE_TYPE.value

reciprocal_vector_ratios = None
if KGRID_CONVERGENCE_TYPE in (ConvergenceParameterNameEnum.N_k_nonuniform, ConvergenceParameterNameEnum.N_k_nonuniform_2D):
    reciprocal_vector_ratios = material.lattice.reciprocal_vector_ratios
    print(f"Reciprocal vector ratios: {reciprocal_vector_ratios}")

convergence_subworkflow = workflow.subworkflows[0]
convergence_subworkflow.add_convergence(
    parameter=CONVERGENCE_PARAMETER,
    parameter_initial=PARAMETER_INITIAL,
    parameter_increment=PARAMETER_INCREMENT,
    reciprocal_vector_ratios=reciprocal_vector_ratios,
    result="total_energy",
    result_initial=RESULT_INITIAL,
    condition="abs((prev_result-total_energy)/total_energy)",
    tolerance=TOLERANCE,
)

print(f"K-grid convergence type: {KGRID_CONVERGENCE_TYPE}")
print(f"Convergence parameter: {convergence_subworkflow.convergence_param}")
print(f"Convergence result: {convergence_subworkflow.convergence_result}")
visualize_workflow(workflow)


## 5. Configure compute
### 5.1. Configure compute


In [ ]:
from mat3ra.ide.compute import Compute

clusters = client.clusters.list()
if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")


## 6. Create the job with material and workflow configuration
### 6.1. Create the job with the embedded convergence workflow


In [ ]:
from utils.api import create_job
from utils.generic import dict_to_namespace
from utils.visualize import display_JSON

job_name = f"{workflow.name} {saved_material.formula}"
job_response = create_job(
    api_client=client,
    materials=[saved_material],
    workflow=workflow,
    project_id=PROJECT_ID,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict(),
)

job = dict_to_namespace(job_response)
job_id = job._id
print(f"Job created: {job_id}")
display_JSON(job_response)


## 7. Submit the job and monitor the status


In [ ]:
client.jobs.submit(job_id)
print(f"Submitted job: {job_id}")


In [ ]:
from utils.api import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)


## 8. Retrieve results
### 8.1. Fetch the finished job and inspect `scopeTrack`


In [ ]:
import matplotlib.pyplot as plt

def plot_convergence_series(series, parameter_name):
    x_labels = [str(item["param"]) for item in series]
    y_values = [item["y"] for item in series]
    x_indices = list(range(len(series)))
    _, ax = plt.subplots()
    ax.plot(x_indices, y_values, marker="o")
    ax.set_xticks(x_indices)
    ax.set_xticklabels(x_labels, rotation=45, ha="right")
    ax.set_xlabel(parameter_name)
    ax.set_ylabel("Total Energy (Ry)")
    ax.set_title(f"K-grid Convergence: {parameter_name}")
    plt.tight_layout()
    plt.show()


subworkflow_index = 0
finished_job = client.jobs.get(job_id)
job_workflow = Workflow.create(finished_job["workflow"])
subworkflow = job_workflow.subworkflows[subworkflow_index]
scope_track = finished_job.get("scopeTrack")
series = subworkflow.convergence_series(scope_track)

parameter_name = job_workflow.subworkflows[subworkflow_index].convergence_param
parameter_value = series[-1]["param"]
if parameter_name == ConvergenceParameterNameEnum.N_k.value:
    converged_kgrid = [parameter_value, parameter_value, parameter_value]
if parameter_name == ConvergenceParameterNameEnum.N_k_nonuniform.value:
    converged_kgrid = list(parameter_value)
if parameter_name == ConvergenceParameterNameEnum.N_k_nonuniform_2D.value:
    converged_kgrid = list(parameter_value)

### 8.2. Reconstruct the convergence series in the notebook
Use `converged_kgrid` in another notebook when setting the SCF k-grid for the production calculation.

In [ ]:
print("Convergence series:")
for item in series:
    print(item)

print(f"\nConvergence parameter: {parameter_name}")
print(f"Converged parameter value: {parameter_value}")
print(f"Converged k-grid: {converged_kgrid}")

plot_convergence_series(series, parameter_name)